# Population Regional CV QC

Regional CV is calculated as the median of the 21 depth-wise column CV values. CI is retained in the cached file but is not displayed. Depth-specific analyses remain in the separate depth-stability notebook.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS=Path('/mnt/newStor/paros/paros_WORK/xintong/results/profile_stability/regional_ci_cv_summary.csv')
OUTPUT=RESULTS.parent
PROFILE_ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/results/column_profiles')
if not RESULTS.is_file() or RESULTS.stat().st_size==0: raise FileNotFoundError(f'Run submit_regional_ci_cv.sbatch first: {RESULTS}')
summary=pd.read_csv(RESULTS); valid=summary[summary.status.eq('valid')].copy()
if 'regional_cv_pct' not in valid: raise RuntimeError('Rerun compute_regional_ci_cv.py to add regional_cv_pct.')
valid['region_label']=valid['hemi']+'_'+valid['region']

def retention_pct(row):
    mask_path=PROFILE_ROOT/row.subject/'profiles'/'bad_trajectory_mask.json'
    if not mask_path.is_file(): return np.nan
    with mask_path.open() as f: entry=json.load(f).get('regions',{}).get(f'{row.hemi}_{row.region}',{})
    removed=int(entry.get('bad_count',len(entry.get('bad_indices',[]))))
    return 100*row.n_trajectories/(row.n_trajectories+removed) if row.n_trajectories+removed else np.nan

print(f'Profiles: {len(summary):,}; valid: {len(valid):,}; subjects: {valid.subject.nunique():,}')

## 1. Overall regional CV by metric

The main plot is limited to the cohort-wide 99th percentile to make the body of the distribution visible. Observations above this limit are written to a separate table.


In [ ]:
overall_limit=float(valid.regional_cv_pct.quantile(.99)); overall_extreme=valid.loc[valid.regional_cv_pct>overall_limit,['subject','hemi','region','metric','regional_cv_pct','mean_value','mean_sd','n_trajectories']].copy(); overall_extreme['retention_pct']=overall_extreme.apply(retention_pct,axis=1); overall_extreme=overall_extreme.sort_values('regional_cv_pct',ascending=False); overall_extreme.to_csv(OUTPUT/'regional_cv_extreme_99th_percentile.csv',index=False)
fig,ax=plt.subplots(figsize=(8,5)); valid.boxplot(column='regional_cv_pct',by='metric',ax=ax,showfliers=False); ax.set(title='Overall regional CV by metric',xlabel='Metric',ylabel='Regional CV (%)',ylim=(0,overall_limit*1.05)); ax.text(.99,.97,f'Display limited to the 99th percentile ({overall_limit:.1f}%)',transform=ax.transAxes,ha='right',va='top',fontsize=9); plt.suptitle(''); fig.tight_layout(); fig.savefig(OUTPUT/'overall_regional_cv_by_metric.png',dpi=180); plt.show(); display(overall_extreme)

## 2. MD regional CV distribution

Each box summarizes all subject × hemisphere observations for one region; regions are ordered by median CV. Individual points are intentionally omitted because 68 dense point clouds obscure the median and IQR. The main display is limited to the MD 99th percentile; values above it are saved separately for QC.


In [ ]:
md=valid[valid.metric.eq('md')].copy(); order=md.groupby('region_label').regional_cv_pct.median().sort_values().index.tolist(); md_limit=float(md.regional_cv_pct.quantile(.99))
plot_data=[md.loc[md.region_label.eq(region),'regional_cv_pct'].dropna().to_numpy() for region in order]
fig,ax=plt.subplots(figsize=(max(14,.23*len(order)),6)); ax.boxplot(plot_data,tick_labels=order,showfliers=False,patch_artist=True,boxprops={'facecolor':'#74a9cf','alpha':.65},medianprops={'color':'#08306b','linewidth':1.5}); ax.set(title='MD regional CV distribution',xlabel='Region',ylabel='Regional CV (%)',ylim=(0,md_limit*1.05)); ax.text(.99,.97,f'Display limited to the 99th percentile ({md_limit:.1f}%)',transform=ax.transAxes,ha='right',va='top',fontsize=9); ax.tick_params(axis='x',rotation=90,labelsize=7); ax.grid(axis='y',alpha=.25); fig.tight_layout(); fig.savefig(OUTPUT/'md_regional_cv_by_region.png',dpi=180); plt.show()
region_summary=md.groupby('region_label',as_index=False).regional_cv_pct.agg(subjects='size',median='median',q25=lambda x:x.quantile(.25),q75=lambda x:x.quantile(.75),minimum='min',maximum='max').sort_values('median',ascending=False); region_summary.to_csv(OUTPUT/'md_regional_cv_summary.csv',index=False)
md_extreme=md.loc[md.regional_cv_pct>md_limit,['subject','hemi','region','region_label','regional_cv_pct','mean_value','mean_sd','n_trajectories']].copy(); md_extreme['retention_pct']=md_extreme.apply(retention_pct,axis=1); md_extreme=md_extreme.sort_values('regional_cv_pct',ascending=False); md_extreme.to_csv(OUTPUT/'md_regional_cv_extreme_99th_percentile.csv',index=False); display(md_extreme); display(region_summary.head(15))

### Bilateral high-CV region summary

This combines left and right homologous regions. The top-quartile repetition rate is the percentage of subjects in which a region falls within that subject's highest 25% of MD regional CV values.


In [ ]:
md_rank=md.copy(); md_rank['subject_q75']=md_rank.groupby('subject').regional_cv_pct.transform(lambda x:x.quantile(.75)); md_rank['top_quartile']=md_rank.regional_cv_pct>=md_rank.subject_q75
hemi_summary=md_rank.groupby(['hemi','region'],as_index=False).agg(subjects=('subject','nunique'),median_cv_pct=('regional_cv_pct','median'),q25_cv_pct=('regional_cv_pct',lambda x:x.quantile(.25)),q75_cv_pct=('regional_cv_pct',lambda x:x.quantile(.75)),top_quartile_subject_pct=('top_quartile',lambda x:100*x.mean()))
wide=hemi_summary.pivot(index='region',columns='hemi',values=['subjects','median_cv_pct','q25_cv_pct','q75_cv_pct','top_quartile_subject_pct'])
wide.columns=[f'{hemi}_{field}' for field,hemi in wide.columns]; bilateral=wide.reset_index().dropna(subset=['lh_median_cv_pct','rh_median_cv_pct']).copy()
bilateral['bilateral_median_cv_pct']=(bilateral.lh_median_cv_pct+bilateral.rh_median_cv_pct)/2; bilateral['bilateral_top_quartile_subject_pct']=(bilateral.lh_top_quartile_subject_pct+bilateral.rh_top_quartile_subject_pct)/2; bilateral=bilateral.sort_values('bilateral_median_cv_pct',ascending=False)
bilateral.to_csv(OUTPUT/'md_bilateral_high_cv_region_summary.csv',index=False)
top=bilateral.head(20).sort_values('bilateral_median_cv_pct'); fig,ax=plt.subplots(figsize=(8,7)); ax.barh(top.region,top.bilateral_median_cv_pct,color='#3182bd'); ax.set(title='Top 20 bilateral regions by median MD regional CV',xlabel='Bilateral median regional CV (%)',ylabel='Region'); ax.grid(axis='x',alpha=.25); fig.tight_layout(); fig.savefig(OUTPUT/'md_bilateral_high_cv_regions.png',dpi=180); plt.show(); display(bilateral.head(20))

## 3. MD subject × region CV heatmap


In [ ]:
heat=md.pivot(index='subject',columns='region_label',values='regional_cv_pct'); heat=heat.loc[heat.median(axis=1).sort_values().index,order]
fig,ax=plt.subplots(figsize=(18,max(8,.06*len(heat)))); im=ax.imshow(heat.to_numpy(),aspect='auto',cmap='magma',vmin=np.nanpercentile(heat,1),vmax=np.nanpercentile(heat,99)); ax.set(title='MD regional CV: subject × region',xlabel='Region',ylabel='Subject'); ax.set_xticks(range(len(order))); ax.set_xticklabels(order,rotation=90,fontsize=6); ax.set_yticks([]); fig.colorbar(im,ax=ax,label='Regional CV (%)'); fig.tight_layout(); fig.savefig(OUTPUT/'md_subject_region_cv_heatmap.png',dpi=180); plt.show()

## 4. MD CV versus cleaned column count

This checks whether regional CV is driven simply by having few retained columns after cleaning.


In [ ]:
rho=md[['regional_cv_pct','n_trajectories']].corr(method='spearman').iloc[0,1]
fig,ax=plt.subplots(figsize=(7,5)); ax.scatter(md.n_trajectories,md.regional_cv_pct,s=4,alpha=.2); ax.set_xscale('log'); ax.set(title=f'MD regional CV versus column count (Spearman rho = {rho:.3f})',xlabel='Column count (log scale)',ylabel='Regional CV (%)'); ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(OUTPUT/'md_cv_vs_column_count.png',dpi=180); plt.show()

## 5. MD CV versus retention percentage

Retention is the proportion of original region columns retained after bad-column cleaning. This tests whether higher regional CV is associated with greater column loss.


In [ ]:
md_retention=md.copy(); md_retention['retention_pct']=md_retention.apply(retention_pct,axis=1); md_retention=md_retention.dropna(subset=['retention_pct'])
rho_retention=md_retention[['regional_cv_pct','retention_pct']].corr(method='spearman').iloc[0,1]
fig,ax=plt.subplots(figsize=(7,5)); ax.scatter(md_retention.retention_pct,md_retention.regional_cv_pct,s=4,alpha=.2); ax.set(title=f'MD regional CV versus retention (Spearman rho = {rho_retention:.3f})',xlabel='Retention (%)',ylabel='Regional CV (%)'); ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(OUTPUT/'md_cv_vs_retention.png',dpi=180); plt.show()
md_retention[['subject','hemi','region','regional_cv_pct','n_trajectories','retention_pct']].to_csv(OUTPUT/'md_cv_retention_summary.csv',index=False)